In [ ]:
%config SqlMagic.autopolars = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [ ]:
%load_ext sql

# Dépendances


In [ ]:
import json
import math
import os
from datetime import datetime, timedelta
from itertools import product
from pathlib import Path
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
import requests
import time

import branca.colormap as bcm
import duckdb
import folium
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars_h3 as plh3
import polars_st as st
import shapely
from dotenv import load_dotenv
from folium import plugins
from sqlalchemy import create_engine

# Configuration


In [ ]:
USE_CACHED_ARTIFACTS = True
USE_CACHED_JOURNEYS_WITH_NEAREST_STATION = True


DEFAULT_START_DATE = datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT"))
ENTITY_CONFIGS = {
    "driver": {
        "identity_col": "driver_identity_key",
        "first_trip_col": "first_trip_datetime",
        "label_plural": "conducteurs",
        "label_singular": "conducteur",
    },
    "passenger": {
        "identity_col": "passenger_identity_key",
        "first_trip_col": "passenger_first_trip_datetime", 
        "label_plural": "passagers",
        "label_singular": "passager",
    }
}

In [ ]:
load_dotenv()
DB_URL = os.environ.get("DB_URL")
IDF_GEOJSON = os.environ.get("IDF_GEOJSON")
BUDGET_IDFM = os.environ.get("BUDGET_IDFM")
DUREE_CAMPAGNE_IDFM = os.environ.get("DUREE_CAMPAGNE_IDFM")

In [ ]:
AOM_SIRET = "28750007800020"

In [ ]:
OUTPUT_PATH = Path("outputs_idfm")

## Labels

In [ ]:
labels_map = {
    "month": "Mois",
    "num_journeys": "Nombre de trajets",
    "share_journeys": "% des trajets",
    "num_journeys_incentived": "Nombre de trajets avec incitation",
    "num_journeys_with_incentive": "Nombre de trajets avec incitation",
    "num_journeys_intra_territory_incentived_trips": "Nombre de trajets incités intra",
    "num_journeys_inter_territory_incentived_trips": "Nombre de trajets incités inter",
    "operator": "Opérateur",
    "incentive_amount_avg": "Incitation moyenne",
    "driver_revenue_avg": "Revenu moyen conducteur",
    "passenger_contribution_avg": "Contribution moyenne passager",
    "incentive_amount_intra_avg": "Incitation moyenne intra",
    "driver_revenue_intra_avg": "Revenu moyen conducteur intra",
    "passenger_contribution_intra_avg": "Contribution moyenne passager intra",
    "incentive_amount_inter_avg": "Incitation moyenne inter",
    "driver_revenue_inter_avg": "Revenu moyen conducteur inter",
    "passenger_contribution_inter_avg": "Contribution moyenne passager inter",
    "incentive_amount_per_km_avg": "Montant moyen d'incitation par km",
    "passenger_contribution_per_km_avg": "Contribution moyenne passager par km",
    "driver_revenue_per_km_avg": "Revenu moyen conducteur par km",
    "day": "Jour",
    "week": "Semaine",
    "month": "Mois",
    "year_month": "Mois",
    "distance_avg": "Distance moyenne",
    "distance_km": "Distance [km]",
    "distance_incentived_trips_avg": "Distance moyenne [km] - trajets avec incentives",
    "campaign_type": "Campagne",
    "distance": "Distance",
    "num_journeys_with_aom_incentive": "Nombre de trajets incités par l'AOM",
    "num_journeys_with_operator_incentive": "Nombre de trajets incités par un opérateur",
    "num_journeys_intra_territory": "Nombre de trajets intra-territoire",
    "num_journeys_inter_territory": "Nombre de trajets inter-territoires",
    "share_journeys_intra_territory": "% de trajets intra-territoire",
    "share_journeys_inter_territory": "% de trajets inter-territoires",
    "share_drivers": "% des conducteurs",
    "num_trips": "Nombre de trips",
    "is_intra_driver": "Conducteur intra",
    "driver_campaign_type": "Type de campagne du conducteur",
    "passenger_campaign_type": "Type de campagne du passager",
    "drivers_share": "% des conducteurs",
    "week_number": "Semaine n°",
    "passengers_share": "% des passagers",
    "num_passenger": "Nombre de passager",
    "is_near_station_fmt": "Catégorie de distance à une gare",
    "has_direct_train_line": "Possède une ligne TC directe",
    "name": "Nom",
    "amount_aom_avg" : "Incitation AOM moyenne",
    "line_name_end":"Ligne",
    "amount_aom": "Incitation AOM",
    "incentive_amount": "Incitation totale",
    "variable": "",
}

import plotly.io as pio

pio.templates.default = "simple_white"
pio.templates[pio.templates.default].layout.font.size = 16

## duckdb


In [ ]:
conn = duckdb.connect(
    "db.duckdb",
    config={"memory_limit": "16GiB", "threads": 4, "preserve_insertion_order": False},
)
%sql conn --alias duckdb

In [ ]:
%%sql
INSTALL spatial;

LOAD spatial;

# Queries


In [ ]:
SQL_ENGINE = create_engine(DB_URL)

## Journeys


In [ ]:
SQL = """
with idfm_perimeter as 
(
select
	p.arr,
	max(p.com) as com,
	max(p.geom_simple) as geom_simple
from
	geo.perimeters p
where
	p.reg = '11'
	and year = 2024
group by
	1
),
geo_filtered as (
select
	g.carpool_id,
	g.start_geo_code,
	g.end_geo_code,
	(substring(g.start_geo_code for 2) in ('94', '78', '92', '91', '75', '93', '95', '77')
		and substring(g.end_geo_code for 2) in ('94', '78', '92', '91', '75', '93', '95', '77')) as is_fully_inside_campaign_area
from
	carpool_v2.geo g
where
	(substring(g.start_geo_code for 2) in ('94', '78', '92', '91', '75', '93', '95', '77')
		or substring(g.end_geo_code for 2) in ('94', '78', '92', '91', '75', '93', '95', '77'))
			and g.updated_at >= '2024-09-01'
),
first_trip as (
select
    driver_identity_key,
    min(c.start_datetime) as first_trip_datetime
from carpool_v2.carpools c
group by 1
),
first_trip_passengers as (
select
    passenger_identity_key,
    min(c.start_datetime) as first_trip_datetime
from carpool_v2.carpools c
group by 1
),
incentives as (
select
	oi.carpool_id,
	sum(oi.amount) as incentive_amount,
    sum(oi.amount) filter (where siret='28750007800020') as amount_aom,
	array_agg(distinct oi.siret) as incentive_sirets
from
	carpool_v2.operator_incentives oi
inner join geo_filtered g on
	oi.carpool_id = g.carpool_id
where amount>0
group by
	1
),
journeys as 
(
select
	c."_id",
	c.operator_id,
	c.operator_journey_id,
	c.operator_trip_id,
    c.driver_identity_key,
    ft.first_trip_datetime,
    c.passenger_identity_key,
    ftp.first_trip_datetime as passenger_first_trip_datetime,
	c.start_datetime,
	c.end_datetime,
	c.distance,
	c.driver_revenue,
	c.passenger_contribution,
	i.incentive_amount,
    i.amount_aom,
	i.incentive_sirets,
	c.start_position,
	c.end_position,
    c.passenger_seats,
    c.passenger_travelpass_name,
    c.passenger_travelpass_user_id,
    is_fully_inside_campaign_area,
	ST_MAKELINE(c.start_position::geometry,c.end_position::geometry) as journey_line,
    t.labels	
from
	carpool_v2.carpools c
inner join geo_filtered g on
	c."_id" = g.carpool_id
left join incentives i on
	c."_id" = i.carpool_id
left join first_trip ft on ft.driver_identity_key=c.driver_identity_key
left join first_trip_passengers ftp on ftp.passenger_identity_key=c.passenger_identity_key
left join carpool_v2.status s on s."carpool_id"=c."_id" 
left join carpool_v2.terms_violation_error_labels t on t."carpool_id"=c."_id" 
where
	(c.start_datetime between '2024-09-01' and '2025-09-30')
    and s.acquisition_status='processed'
    and s.fraud_status='passed'
    and s.anomaly_status='passed'
    )
SELECT
    j.*,
    CASE WHEN p.l_arr = p.country THEN p.l_country ELSE p.l_arr END as start_com,
    CASE WHEN p2.l_arr = p2.country THEN p2.l_country ELSE p2.l_arr END as end_com
from journeys j
left join carpool_v2.geo g on j."_id"=g."carpool_id"
left join geo.perimeters p on g."start_geo_code"=p.arr and p.year=2024
left join geo.perimeters p2 on g."end_geo_code"=p2.arr and p2.year=2024
"""

In [ ]:
if USE_CACHED_ARTIFACTS:
    df_journeys_raw = pl.read_parquet("df_journeys_raw_idfm.parquet")
else:
    df_journeys_raw = pl.read_database(
        query=SQL,
        connection=SQL_ENGINE,
        schema_overrides={
            "passenger_travelpass_name": pl.String,
            "passenger_travelpass_user_id": pl.String,
            "labels": pl.List(pl.String),
        },
    )
    df_journeys_raw.write_parquet("df_journeys_raw_idfm.parquet", compression_level=6)

In [ ]:
df_journeys_raw.schema

In [ ]:
df_journeys_raw.estimated_size() / 1e7

In [ ]:
df_journeys_raw.head()

In [ ]:
df_journeys_raw.describe()

## Opérateurs


In [ ]:
df_operators = pl.read_database(
    query="""
SELECT
    "_id",
    "name",
    "siret"
from operator.operators
where deleted_at is null
and name!='BlaBlaCar'
""",
    connection=SQL_ENGINE,
)

In [ ]:
df_operators

In [ ]:
df_karos=pl.DataFrame({"_id":[999], "name":["Karos"], "siret":["80279897500024"]})
df_operators = pl.concat([df_operators, df_karos])
df_operators

# Reseau IDFM


In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS gares_idfm AS
SELECT
  id_gares,
  nom_gares,
  nom_so_gar,
  nom_su_gar,
  id_ref_zdc,
  nom_zdc,
  id_ref_zda,
  nom_zda,
  idrefliga,
  idrefligc,
  res_com,
  indice_lig,
  mode,
  tertrain,
  terrer,
  termetro,
  tertram,
  terval,
  exploitant,
  idf,
  ST_FlipCoordinates (geom) AS geom --  EPSG:4326 coordinate system (WGS84), with [latitude, longitude] axis order
FROM
    ST_Read('{{IDF_GEOJSON}}')
WHERE
  mode IN ('TRAIN', 'RER');

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  gares_idfm;

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS gares_geom_index ON gares_idfm USING RTREE (geom);

In [ ]:
%%sql
FROM
  (DESCRIBE gares_idfm);

In [ ]:
%%sql df_idfm_stations <<
SELECT
    *,
    ST_asText(geom) as geom_wkt
FROM gares_idfm

In [ ]:
df_idfm_stations

# Identification incitateurs


In [ ]:
df_journeys_raw = df_journeys_raw.with_columns(
    pl.col("incentive_sirets").list.contains(AOM_SIRET).alias("incentived_by_aom"),
    (
        pl.col("incentive_sirets")
        .list.set_intersection(df_operators["siret"].to_list())
        .list.len()
        > 0
    ).alias("incentived_by_operator"),
)

# Traitements geo


In [ ]:
df_journeys_raw = df_journeys_raw.with_columns(
        pl.col("start_position")
        .map_elements(lambda x: shapely.from_wkb(x).wkt, return_dtype=pl.String)
        .alias("start_pos"),
        pl.col("end_position")
        .map_elements(lambda x: shapely.from_wkb(x).wkt, return_dtype=pl.String)
        .alias("end_pos"),
        pl.col("start_position")
        .map_elements(lambda x: shapely.from_wkb(x).y, return_dtype=pl.Float64)
        .alias("start_latitude"),
        pl.col("start_position")
        .map_elements(lambda x: shapely.from_wkb(x).x, return_dtype=pl.Float64)
        .alias("start_longitude"),
            pl.col("end_position")
        .map_elements(lambda x: shapely.from_wkb(x).y, return_dtype=pl.Float64)
        .alias("end_latitude"),
        pl.col("end_position")
        .map_elements(lambda x: shapely.from_wkb(x).x, return_dtype=pl.Float64)
        .alias("end_longitude"),
    )

# Filtrage des journeys sans incitations


In [ ]:
df_journeys = df_journeys_raw.filter((pl.col("incentive_amount") > 0))

# Création de la table des journeys sur duckdb


In [ ]:
%%sql
CREATE TABLE
  if NOT EXISTS journeys_raw AS
SELECT
  _id,
  operator_id,
  operator_journey_id,
  operator_trip_id,
  driver_identity_key,
  first_trip_datetime,
  passenger_identity_key,
  passenger_first_trip_datetime,
  start_datetime,
  end_datetime,
  distance,
  driver_revenue,
  passenger_contribution,
  incentive_amount,
  amount_aom,
  incentive_sirets,
  start_position,
  end_position,
  passenger_seats,
  is_fully_inside_campaign_area,
  journey_line,
  start_com,
  end_com,
  incentived_by_aom,
  incentived_by_operator,
  ST_FlipCoordinates (ST_GeomFromText (start_pos)) AS start_pos,
  ST_FlipCoordinates (ST_GeomFromText (end_pos)) AS end_pos
FROM
  df_journeys_raw

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS start_pos_idx ON journeys_raw USING RTREE (start_pos);

CREATE INDEX IF NOT EXISTS end_pos_idx ON journeys_raw USING RTREE (end_pos);

In [ ]:
%%sql
FROM
  (DESCRIBE journeys_raw)

In [ ]:
%%sql
SELECT
  *
FROM
  journeys_raw
LIMIT
  5

In [ ]:
%%sql
SELECT
  COUNT(*)
FROM
  journeys_raw

# Statistiques globales


In [ ]:
incentived_trip_filter_expr = pl.col("incentive_amount") > 0

agg_expressions = [
    pl.col("_id").n_unique().alias("num_journeys"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area"))
    .n_unique()
    .alias("num_journeys_intra_territory"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
    .n_unique()
    .alias("num_journeys_intra_territory_incentived_trips"),
    pl.col("_id")
    .filter(incentived_trip_filter_expr)
    .n_unique()
    .alias("num_journeys_incentived"),
    pl.col("_id")
    .filter(pl.col("incentived_by_aom"))
    .n_unique()
    .alias("num_journeys_with_aom_incentive"),
    pl.col("_id")
    .filter(pl.col("incentived_by_operator"))
    .n_unique()
    .alias("num_journeys_with_operator_incentive"),
    pl.col("_id")
    .filter(pl.col("is_fully_inside_campaign_area"))
    .n_unique()
    .alias("num_journeys_intra"),
    (pl.col("distance") / 1000).mean().alias("distance_avg"),
    (pl.col("distance").filter(incentived_trip_filter_expr) / 1000)
    .mean()
    .alias("distance_incentived_trips_avg"),
    (pl.col("amount_aom").sum()/100).alias("amount_aom_sum"),
    (pl.col("amount_aom").mean()/100).alias("amount_aom_avg"),
    (pl.col("incentive_amount").mean() / 100).alias("incentive_amount_avg"),
    (pl.col("passenger_contribution").filter(incentived_trip_filter_expr.not_()) / 100)
    .mean()
    .alias("passenger_contribution_avg"),
    (pl.col("passenger_contribution").filter(incentived_trip_filter_expr) / 100)
    .mean()
    .alias("passenger_contribution_incentived_trips_avg"),
    (
        pl.col("driver_revenue").filter(incentived_trip_filter_expr.not_()).mean() / 100
    ).alias("driver_revenue_avg"),
    (pl.col("driver_revenue").filter(incentived_trip_filter_expr).mean() / 100).alias(
        "driver_revenue_incentived_trips_avg"
    ),
    (
        pl.col("incentive_amount")
        .filter(pl.col("is_fully_inside_campaign_area"))
        .mean()
        / 100
    ).alias("incentive_amount_intra_avg"),
    (
        pl.col("passenger_contribution")
        .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
        .mean()
        / 100
    ).alias("passenger_contribution_intra_avg"),
    (
        pl.col("driver_revenue")
        .filter(pl.col("is_fully_inside_campaign_area") & incentived_trip_filter_expr)
        .mean()
        / 100
    ).alias("driver_revenue_intra_avg"),
    (
        pl.col("incentive_amount")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("incentive_amount_inter_avg"),
    (
        pl.col("passenger_contribution")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("passenger_contribution_inter_avg"),
    (
        pl.col("driver_revenue")
        .filter(
            pl.col("is_fully_inside_campaign_area").not_() & incentived_trip_filter_expr
        )
        .mean()
        / 100
    ).alias("driver_revenue_inter_avg"),
    (10 * (pl.col("incentive_amount") / pl.col("distance")))
    .mean()
    .alias("incentive_amount_per_km_avg"),
    (10 * (pl.col("amount_aom") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        ))
    .mean()
    .alias("aom_amount_per_km_avg"),    
    (
        10
        * (pl.col("passenger_contribution") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("passenger_contribution_per_km_avg"),
    (
        10
        * (pl.col("passenger_contribution") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("passenger_contribution_per_km_incentived_trips_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr.not_() &  pl.col("is_fully_inside_campaign_area").not_()
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg_inter"),
     (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr.not_() &  pl.col("is_fully_inside_campaign_area")
        )
    )
    .mean()
    .alias("driver_revenue_per_km_avg_intra"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr &  pl.col("is_fully_inside_campaign_area").not_()
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg_inter"),
    (
        10
        * (pl.col("driver_revenue") / pl.col("distance")).filter(
            incentived_trip_filter_expr &  pl.col("is_fully_inside_campaign_area")
        )
    )
    .mean()
    .alias("driver_revenue_per_km_incentived_trips_avg_intra"),
    pl.col("driver_identity_key").n_unique().alias("number_of_unique_driver"),
    pl.col("passenger_identity_key").n_unique().alias("number_of_unique_passenger"),

]

In [ ]:
df_stats_by_month = (
    df_journeys_raw.group_by(pl.col("start_datetime").dt.truncate("1mo").alias("month"))
    .agg(agg_expressions)
    .sort(pl.col("month"))
)

In [ ]:
df_stats_by_week = (
    df_journeys_raw.filter(
        pl.col("start_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT"))
    )
    .group_by(pl.col("start_datetime").dt.truncate("1w").alias("week"))
    .agg(agg_expressions)
    .sort(pl.col("week"))
)

df_stats_by_week_rm_citygo = (
    df_journeys_raw.filter(
        pl.col("start_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT")), pl.col("operator_id") != 274
    )
    .group_by(pl.col("start_datetime").dt.truncate("1w").alias("week"))
    .agg(agg_expressions)
    .sort(pl.col("week"))
)

## Nombre de journeys


In [ ]:
with pl.Config(set_fmt_str_lengths=120, set_tbl_width_chars=1000):
    print(
        df_journeys_raw.select(
            pl.col("_id").n_unique().alias("Nombre de trajets"),
            pl.col("_id")
            .filter(pl.col("incentive_amount") > 0)
            .n_unique()
            .alias("Nombre de trajets avec incitation"),
            (
                100
                * pl.col("_id").filter(pl.col("incentive_amount") > 0).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% trajets avec incitation"),
            pl.col("_id")
            .filter(pl.col("incentived_by_aom"))
            .n_unique()
            .alias("Nombre de trajets avec incitation AOM"),
            (
                100
                * pl.col("_id").filter(pl.col("incentived_by_aom")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% trajets avec incitation AOM"),
            pl.col("_id")
            .filter(pl.col("incentived_by_operator"))
            .n_unique()
            .alias("Nombre de trajets avec incitation opérateur"),
            (
                100
                * pl.col("_id").filter(pl.col("incentived_by_operator")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% trajets avec incitation opérateur"),



                        pl.col("_id")
            .filter(pl.col("incentived_by_operator"), ~pl.col("incentived_by_aom"))
            .n_unique()
            .alias("Nombre de trajets avec incitation opérateur seule"),
(
    100.0
    * pl.col("_id").filter((pl.col("incentived_by_operator")) & (~pl.col("incentived_by_aom"))).n_unique()
    / pl.col("_id").n_unique()
).alias("% trajets avec incitation opérateur seule")

        )
        .with_columns(pl.selectors.all().round(2))
        .unpivot()
    )

In [ ]:
df_analyzed = df_journeys_raw.with_columns([
    pl.col("start_datetime").dt.date().alias("trip_date"),
    pl.col("start_datetime").dt.year().alias("trip_year"),
    pl.col("start_datetime").dt.month().alias("trip_month"),
    
    pl.col("start_datetime")
        .rank("ordinal")
        .over(["driver_identity_key", pl.col("start_datetime").dt.date()])
        .alias("trip_rank_in_day"),
    
    pl.col("amount_aom")
        .fill_null(0)
        .cum_sum()
        .over(["driver_identity_key", pl.col("start_datetime").dt.year(), pl.col("start_datetime").dt.month()], 
              order_by="start_datetime")
        .alias("cumulative_aom_incentive_month")
])

df_analyzed = df_analyzed.with_columns([
    (pl.col("trip_rank_in_day") > 6).alias("exceeds_daily_limit"),
    (pl.col("cumulative_aom_incentive_month") >= 15000).alias("exceeds_monthly_limit")
])

with pl.Config(set_fmt_str_lengths=120, set_tbl_width_chars=1000):
    print(
        df_analyzed.select([
            (
                100
                * pl.col("_id").filter(pl.col("amount_aom").is_null()).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys n'ayant pas reçu d'incitation aom"),
            
            (
                100
                * pl.col("_id").filter(pl.col("distance") < 2000).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys avec distance < 2 km"),
            
            (
                100
                * pl.col("_id").filter(~pl.col("is_fully_inside_campaign_area")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys étant en partie hors du périmètre"),
            
            (
                100
                * pl.col("_id").filter(pl.col("labels").list.contains("too_many_trips_by_day")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys avec trop de trajets par jour"),
            
            (
                100
                * pl.col("_id").filter(pl.col("labels").list.contains("too_close_trips")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys avec trajets trop rapprochés"),
             (
                100
                * pl.col("_id").filter(pl.col("labels").list.contains("expired")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys ont expiré"),    

            (
                100
                * pl.col("_id").filter(pl.col("labels").list.contains("distance_too_short")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys sont trop courts"),    

            (
                100
                * pl.col("_id").filter(pl.col("exceeds_daily_limit")).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys dépassant 6 trajets/jour"),
            
            (
                100
                * pl.col("_id").filter(pl.col("operator_id")==274).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys citygo"),
            (
                100
                * pl.col("_id").filter(pl.col("incentive_amount").is_null()).n_unique()
                / pl.col("_id").n_unique()
            ).alias("% journeys n'ont pas du tout d'incentive"),
        ])
        .with_columns(pl.selectors.all().round(2))
        .unpivot()
    )

### Evolution


#### Globale


In [ ]:
def create_num_journeys_fig(
    df: pl.DataFrame,
    x_col: str = "month",
    title: str = "IDFM - Nombre de trajets par mois",
) -> go.Figure:
    traces = []
    max_y = 0

    df_grouped = df.group_by()
    for name in [
        "num_journeys",
        "num_journeys_incentived",
        "num_journeys_with_aom_incentive",
        "num_journeys_with_operator_incentive",
    ]:
        trace = go.Scatter(
            x=df[x_col],
            y=df[name],
            mode="lines+text" if name == "num_journeys" else "lines",
            textposition="top center",
            text=df[name] if name == "num_journeys" else None,
            name=labels_map.get(name, name),
        )
        traces.append(trace)
        max_y = max(max_y, df[name].max())

    fig = go.Figure(traces)

    fig.update_layout(
        template="simple_white",
        title=title,
        legend_orientation="h",
        legend_y=0.6,
        legend_yref="container",
    )
    fig.update_xaxes(title="Mois" if x_col == "month" else "Semaine")
    fig.update_yaxes(range=[0, max_y * 1.2], showgrid=True, title="Nombre de trajets")

    return fig

In [ ]:
fig_journeys_by_month = create_num_journeys_fig(df_stats_by_month)
fig_journeys_by_month.show()

fig_journeys_by_month.write_html(OUTPUT_PATH / "fig_journeys_par_mois.html")
fig_journeys_by_month.write_image(
    OUTPUT_PATH / "fig_journeys_par_mois.svg", width=1280, height=720
)

#### Opérateur incitateurs


In [ ]:
fig_journeys_by_operator = px.line(
    df_journeys.explode("incentive_sirets")
    .join(df_operators, left_on="incentive_sirets", right_on="siret", how="left")
    .group_by(["name", pl.col("start_datetime").dt.truncate("1mo")])
    .agg(pl.col("operator_journey_id").n_unique().alias("num_journeys"))
    .rename({"name": "operator", "start_datetime": "month"})
    .sort("month"),
    x="month",
    y="num_journeys",
    color="operator",
    template="simple_white",
    labels=labels_map,
    title="Nombre de journeys incités par opérateur",
)
fig_journeys_by_operator.update_yaxes(showgrid=True)
fig_journeys_by_operator.show()


fig_journeys_by_operator.write_html("outputs_idfm/fig_journeys_par_operateur_mois.html")
fig_journeys_by_operator.write_image(
    "outputs_idfm/fig_journeys_par_operateur_mois.svg", width=1280, height=720
)

#### intra vs inter


In [ ]:
fig_journeys_by_journey_type = px.line(
    df_stats_by_week.with_columns(
        (pl.col("num_journeys") - pl.col("num_journeys_intra_territory")).alias(
            "num_journeys_inter_territory"
        )
    ),
    x="week",
    y=["num_journeys_intra_territory", "num_journeys_inter_territory"],
    template="simple_white",
    labels=labels_map,
    title="Nombre de trajets par type de trajets",
)
fig_journeys_by_journey_type.update_traces(
    {"name": labels_map["num_journeys_intra_territory"]},
    selector={"name": "num_journeys_intra_territory"},
)
fig_journeys_by_journey_type.update_traces(
    {"name": labels_map["num_journeys_inter_territory"]},
    selector={"name": "num_journeys_inter_territory"},
)
fig_journeys_by_journey_type.update_yaxes(showgrid=True, title="Nombre de trajets")
fig_journeys_by_journey_type.update_layout(
    legend_title="", legend_orientation="h", legend_y=0.7, legend_yref="container"
)
fig_journeys_by_journey_type.show()


fig_journeys_by_journey_type.write_html(
    OUTPUT_PATH / "fig_journeys_par_type_semaine.html"
)
fig_journeys_by_journey_type.write_image(
    OUTPUT_PATH / "fig_journeys_par_type_semaine.svg", width=1280, height=720
)

## Par opérateurs


In [ ]:
fig_journeys_by_operator = px.line(
    df_journeys.group_by(["operator_id", pl.col("start_datetime").dt.truncate("1w")])
    .agg(pl.col("_id").n_unique().alias("num_journeys"))
    .join(df_operators, left_on="operator_id", right_on="_id", validate="m:1")
    .sort(["start_datetime", "operator_id"]),
    x="start_datetime",
    y="num_journeys",
    color="name",
    labels=labels_map,
    template="simple_white",
    title="Nombre de journeys par opérateur par semaine <br><sub>Uniquement les trajets incités</sub>",
)

fig_journeys_by_operator.write_html(
    OUTPUT_PATH / "fig_journeys_by_operator.html"
)
fig_journeys_by_operator.write_image(
    OUTPUT_PATH / "fig_journeys_by_operator.svg", width=1280, height=720
)
fig_journeys_by_operator.show()

## Distance


In [ ]:
fig_distance_by_month = px.box(
    df_journeys_raw.with_columns(
        (pl.col("distance") / 1000).alias("distance_km"), pl.col("start_datetime").dt.strftime("%Y-%m").alias("year_month")
    ),
    x="year_month",
    y="distance_km",
    # log_y=True,
    # text="distance_avg_fmt",
    template="simple_white",
    labels=labels_map,
    title="Distribution de la distance par trajet par mois - tous trajets",
    points="outliers",
    
)
# fig_distance_by_month.update_traces(quartilemethod="exclusive")
fig_distance_by_month.update_yaxes(
    range=[(df_journeys_raw["distance"].min()*.9)/1000., 70], zeroline=True
)
fig_distance_by_month.update_yaxes(showgrid=True)
fig_distance_by_month.update_traces(marker=dict(size=0, opacity=0))
# fig_distance_by_month.show()
fig_distance_by_month.write_html(OUTPUT_PATH / "fig_distance_par_mois.html")
fig_distance_by_month.write_image(
    OUTPUT_PATH / "fig_distance_par_mois.svg", width=1280, height=720
)

## Prix, revenus et incitations


### Trajets Incités


In [ ]:
def create_scatter_fig_prices(
    df: pl.DataFrame,
    stats_cols: list[str],
    x_col: str,
    title: str,
    labels_map: dict[str, str],
    x_title: str = "Montant (euros)",
    min_y: float = 0,
    max_y: float = None
) -> go.Figure:
    traces = []
    for name in stats_cols:
        trace = go.Scatter(
            x=df[x_col],
            y=df[name],
            name=labels_map.get(name, name),
            mode="lines+markers",
            marker_size=4,
        )
        traces.append(trace)
    fig = go.Figure(traces)
    fig.update_layout(
        template="simple_white",
        title=title,
        legend_orientation="h",
        legend_y=0.7,
        legend_yref="container",
    )

    if not max_y:
        max_y = df.select(stats_cols).max().max_horizontal().item()

    fig.update_yaxes(
        range=[min_y, max_y * 1.2],
        title=x_title,
        showgrid=True,
        gridwidth=2,
        ticksuffix="€",
    )
    fig.update_xaxes(title="Mois" if x_col == "month" else "Semaine")

    return fig

In [ ]:
fig_prices_by_week = create_scatter_fig_prices(
    df_stats_by_week,
    [
        "incentive_amount_avg",
        "passenger_contribution_incentived_trips_avg",
        "driver_revenue_incentived_trips_avg",
        "amount_aom_avg"
    ],
    "week",
    (
        "Montants moyens par trajet des incitations,"
        "<br>contributions passagers et revenus conducteurs - Trajets incités"
    ),
    {
        **labels_map,
        "passenger_contribution_incentived_trips_avg": "Contribution moyenne passager",
        "driver_revenue_incentived_trips_avg": "Revenu moyen conducteur",
    },
)
fig_prices_by_week.show()


fig_prices_by_week.write_html(OUTPUT_PATH / "fig_prix_par_semaine.html")
fig_prices_by_week.write_image(
    OUTPUT_PATH / "fig_prix_par_semaine.svg", width=1280, height=720
)

In [ ]:
df_june = df_journeys.filter(
    (pl.col("start_datetime") >= pl.date(2025, 6, 1)) &
    (pl.col("start_datetime") <= pl.date(2025, 6, 28)) &
    (pl.col("amount_aom") < 1000) &
    (pl.col("incentive_amount") > 0)
)

fig_incenitves_distrib = go.Figure()

fig_incenitves_distrib.add_trace(go.Histogram(
    x=df_june["amount_aom"].to_list(),
    nbinsx=100,
    name="Incitation AOM",
    marker_color='red',
    opacity=0.7,
    xbins=dict(start=0, end=500, size=5)
))

fig_incenitves_distrib.add_trace(go.Histogram(
    x=df_june["incentive_amount"].to_list(),
    nbinsx=100,
    name="Incitation totale",
    marker_color='blue',
    opacity=0.7,
    xbins=dict(start=0, end=500, size=5)
))

fig_incenitves_distrib.update_layout(
    template="simple_white",
    title="Distribution de l'incitation totale reçue par trajet - juin 2025",
    xaxis_title="Incitations [€]",
    yaxis_title="Nombre de trajets",
    height=500,
    barmode='group'
)
fig_incenitves_distrib.show()


fig_incenitves_distrib.write_html(OUTPUT_PATH / "fig_incenitves_distrib.html")
fig_incenitves_distrib.write_image(
    OUTPUT_PATH / "fig_incenitves_distrib.svg", width=1280, height=720
)

In [ ]:

fig_cdf = px.ecdf(
    df_june.to_pandas(),
    x=["amount_aom", "incentive_amount"],
    template="simple_white",
    title="Distribution cumulative - juin 2025",
    labels=labels_map,
    height=500,
    color_discrete_map={
        "amount_aom": "red",
        "incentive_amount": "blue"
    }
)

fig_cdf.update_layout(
    xaxis_title="Incitations [€]",
    yaxis_title="Probabilité",
    
)

fig_cdf.write_html(OUTPUT_PATH / "fig_incenitves_cumulative.html")
fig_cdf.write_image(
    OUTPUT_PATH / "fig_incenitves_cumulative.svg", width=1280, height=720
)

#### Intra


In [ ]:
fig_prices_by_week_intra = create_scatter_fig_prices(
    df_stats_by_week,
    [
        "incentive_amount_intra_avg",
        "passenger_contribution_intra_avg",
        "driver_revenue_intra_avg",
    ],
    "week",
    (
        "Montants moyens par trajet <b>intra</b> des incitations,"
        "<br>contributions passagers et revenus conducteurs"
    ),
    labels_map,
)
fig_prices_by_week_intra.show()


fig_prices_by_week_intra.write_html(OUTPUT_PATH / "fig_prix_intra_par_semaine.html")
fig_prices_by_week_intra.write_image(
    OUTPUT_PATH / "fig_prix_intra_par_semaine.svg", width=1280, height=720
)

#### Inter


In [ ]:
fig_prices_by_week_inter = create_scatter_fig_prices(
    df_stats_by_week,
    [
        "incentive_amount_inter_avg",
        "passenger_contribution_inter_avg",
        "driver_revenue_inter_avg",
    ],
    "week",
    (
        "Montants moyens par trajet <b>inter</b> des incitations,"
        "<br>contributions passagers et revenus conducteurs"
    ),
    labels_map,
)
fig_prices_by_week_inter.show()


fig_prices_by_week_inter.write_html(OUTPUT_PATH / "fig_prix_inter_par_semaine.html")
fig_prices_by_week_inter.write_image(
    OUTPUT_PATH / "fig_prix_inter_par_semaine.svg", width=1280, height=720
)

#### Au kilomètre


In [ ]:
fig_prices_per_km_by_week = create_scatter_fig_prices(
    df_stats_by_week,
    [
        "incentive_amount_per_km_avg",
        "passenger_contribution_per_km_incentived_trips_avg",
        "driver_revenue_per_km_incentived_trips_avg",
         "aom_amount_per_km_avg"
   ],
    "week",
    (
        "Montants moyens <b>par km</b> des incitations,"
        "contributions passagers et revenus conducteurs"
        "<br><sub>Uniquement les trajets incités</sub>"
    ),
    {
        **labels_map,
        "incentive_amount_per_km_avg": "Incitation moyenne",
        "passenger_contribution_per_km_incentived_trips_avg": "Contribution moyenne passager",
        "driver_revenue_per_km_incentived_trips_avg": "Revenu moyen conducteur",
        "aom_amount_per_km_avg": "Montant moyen d'incitation AOM"
    },
    x_title="Montant (euros/km)",
)
fig_prices_per_km_by_week.show()


fig_prices_per_km_by_week.write_html(OUTPUT_PATH / "fig_prix_par_km_par_semaine.html")
fig_prices_per_km_by_week.write_image(
    OUTPUT_PATH / "fig_prix_par_km_par_semaine.svg", width=1280, height=720
)

In [ ]:
def create_scatter_fig_prices_inter_vs_intra(
    df: pl.DataFrame,
    stats_cols: list[str],
    x_col: str,
    title: str,
    labels_map: dict[str, str],
    x_title: str = "Montant (euros)",
    min_y: float = 0,
    max_y: float = None
) -> go.Figure:  
    
    if not max_y:
        max_y = df.select(stats_cols).max().max_horizontal().item()
    
    df = df.unpivot(
        index=[x_col, "is_fully_inside_campaign_area"],
        on=stats_cols,
        variable_name="metric",
        value_name="value"
    )
    
    df = df.to_pandas()
    
    fig = px.line(
        df,
        x=x_col,  
        y="value",  
        color="metric",  
        line_dash="is_fully_inside_campaign_area",
        labels=labels_map,
        template="simple_white",
    )
    for trace in fig.data:
        original_name = trace.name.split(",")[0]
        if original_name in labels_map:
            trace.name = trace.name.replace(original_name, labels_map[original_name])
    
    fig.update_layout(
        title=title,

    )
    
    fig.update_yaxes(
        range=[min_y, max_y * 1.2],
        title=x_title,
        showgrid=True,
        gridwidth=2,
        ticksuffix="€",
    )
    
    fig.update_xaxes(title="Mois" if x_col == "month" else "Semaine")
    
    return fig

In [ ]:
df_stats_by_week_intra_vs_inter = (
    df_journeys_raw.filter(
        pl.col("start_datetime") <= datetime(2025, 7, 31, tzinfo=ZoneInfo("GMT"))
    )
    .group_by(pl.col("start_datetime").dt.truncate("1w").alias("week"), "is_fully_inside_campaign_area")
    .agg(agg_expressions)
    .sort(pl.col("week"))
)

In [ ]:
fig_prices_per_km_by_week_intra_vs_inter = create_scatter_fig_prices_inter_vs_intra(
    df_stats_by_week_intra_vs_inter,
    [
        "incentive_amount_per_km_avg",
        "passenger_contribution_per_km_incentived_trips_avg",
        "driver_revenue_per_km_incentived_trips_avg",
    ],
    "week",
    (
        "Montants moyens <b>par km</b> des incitations,"
        "contributions passagers et revenus conducteurs"
        "<br><sub>Uniquement les trajets incités</sub>"
    ),
    {
        **labels_map,
        "incentive_amount_per_km_avg": "Incitation moyenne",
        "passenger_contribution_per_km_incentived_trips_avg": "Contribution moyenne passager",
        "driver_revenue_per_km_incentived_trips_avg": "Revenu moyen conducteur",
    },
    x_title="Montant (euros/km)",
)
fig_prices_per_km_by_week_intra_vs_inter.show()


fig_prices_per_km_by_week_intra_vs_inter.write_html(OUTPUT_PATH / "fig_prix_par_km_par_semaine_inter_vs_intra.html")
fig_prices_per_km_by_week_intra_vs_inter.write_image(
    OUTPUT_PATH / "fig_prix_par_km_par_semaine_inter_vs_intra.svg", width=1280, height=720
)

### Incitation par rapport à la distance


In [ ]:
px.scatter(
    df_journeys.filter(
        pl.col("amount_aom").is_not_null()
        & (pl.col("incentive_sirets") == [AOM_SIRET])
        & (pl.col("is_fully_inside_campaign_area").not_())
    )
    .with_columns(pl.col("distance") / 1000)
    .unpivot(
        on=["amount_aom", "passenger_contribution", "driver_revenue"],
        index=["_id", "distance"],
    )
    .sort("distance"),
    x="distance",
    y="value",
    color="variable",
    template="simple_white",
)

In [ ]:
df_incentive_aom_by_distance = (
    df_journeys.filter(
        (pl.col("incentive_sirets") == [AOM_SIRET])
        & (pl.col("is_fully_inside_campaign_area")) & (pl.col("amount_aom")<= 450)
    )
    .with_columns(
        pl.col("distance") / 1000,
        pl.col("driver_revenue")
        .cum_sum()
        .over(
            partition_by=[
                "driver_identity_key",
                pl.col("start_datetime").dt.truncate("1mo"),
            ],
            order_by="start_datetime",
        )
        .alias("driver_revenue_cumsum"),
    )
    .filter(pl.col("driver_revenue_cumsum") <= 5000)
    .group_by(
        [
            pl.col("distance").cut(
                list(range(0, 100, 5)), include_breaks=True, left_closed=True
            ),
        ]
    )
    .agg(
        pl.len(),
        (pl.col("amount_aom") / 100).mean().alias("Incitation AOM"),
        (pl.col("passenger_contribution") / 100).alias("Contribution passager").mean(),
        (pl.col("driver_revenue") / 100).alias("Revenu conducteur").mean(),
    )
    .with_columns(pl.col("distance").struct.unnest())
    .unpivot(
        on=["Incitation AOM", "Contribution passager", "Revenu conducteur"],
        index=[
            "breakpoint",
            "category",
        ],
    )
    .sort(["breakpoint"])
)

In [ ]:
fig_incentive_aom_by_distance = px.line(
    df_incentive_aom_by_distance,
    x="breakpoint",
    y="value",
    color="variable",
    template="simple_white",
    height=800,
    labels={**labels_map, "breakpoint": "Distance", "value": "Montant (€)"},
)

fig_incentive_aom_by_distance.update_layout(
    legend_title="",
    title="Montants moyens du revenu conducteur, contribution passager et incitation AOM en fonction de la distance"
    "<br><sub>Uniquement les trajets inter, intervalles de distance de 5km.</sub>",
)
fig_incentive_aom_by_distance.show()

fig_incentive_aom_by_distance.write_html(
    "outputs_idfm/fig_incitation_aom_par_distance.html"
)
fig_incentive_aom_by_distance.write_image(
    "outputs_idfm/fig_incitation_aom_par_distance.svg", width=1280, height=720
)

# Conducteurs


In [ ]:
print(
    f"Nombre de conducteurs uniques : {
        df_journeys_raw.select(pl.col('driver_identity_key').n_unique()).item()
    }"
)

In [ ]:
def create_acquisition_chart(
    df: pl.DataFrame,
    entity_type: str,
    start_date: datetime = DEFAULT_START_DATE,
    labels_map: dict = None,
    window: str = "1w",
    title_prefix: str = "Evolution de l'acquisition",
    group_by_col: str | None = None,
    color_discrete_map: dict | None = None,
    barmode: str = "group",
) -> go.Figure:
    """
    Crée un graphique d'acquisition par période avec groupement optionnel.
    
    Args:
        df: DataFrame Polars
        entity_type: "driver" ou "passenger"
        start_date: Date de début du filtre (par défaut 1er sept 2024)
        labels_map: Dictionnaire de mapping des labels
        window: Fenêtre temporelle ("1d", "1w", "1mo")
        title_prefix: Préfixe du titre
        group_by_col: Colonne optionnelle pour grouper (ex: "is_near_station_fmt")
        color_discrete_map: Mapping des couleurs pour le groupement
        barmode: Mode d'affichage des barres ("group", "stack", "relative")
    
    Returns:
        Figure plotly
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    datetime_col = config["first_trip_col"]
    entity_label = config["label_plural"]
    
    group_cols = [pl.col(datetime_col).dt.truncate(window).alias("period")]
    if group_by_col:
        group_cols.append(group_by_col)
    
    sort_cols = ["period"]
    if group_by_col:
        sort_cols.append(group_by_col)
    
    chart_df = (
        df.filter(pl.col(datetime_col) >= start_date)
        .group_by(group_cols)
        .agg(pl.len())
        .sort(sort_cols)
    )
    
    if labels_map is None:
        labels_map = {}
    
    chart_labels = {**labels_map, "len": f"Nombre de nouveaux {entity_label}"}
    
    fig_kwargs = {
        "data_frame": chart_df,
        "x": "period",
        "y": "len",
        "labels": chart_labels,
        "template": "simple_white",
        "title": f"{title_prefix} des {entity_label}",
    }
    
    if group_by_col:
        fig_kwargs["color"] = group_by_col
        fig_kwargs["barmode"] = barmode
        if color_discrete_map:
            fig_kwargs["color_discrete_map"] = color_discrete_map
    
    fig = px.bar(**fig_kwargs)
    
    return fig

In [ ]:
fig_new_drivers_count_by_week = create_acquisition_chart(
    df=df_journeys_raw,
    entity_type="driver",
    labels_map=labels_map
)


fig_new_drivers_count_by_week.update_yaxes(showgrid=True)

fig_new_drivers_count_by_week.show()

fig_new_drivers_count_by_week.write_html(
    OUTPUT_PATH / "fig_conducteurs_par_semaine.html"
)
fig_new_drivers_count_by_week.write_image(
    OUTPUT_PATH / "fig_conducteurs_par_semaine.svg", width=1280, height=720
)

## Nombre de trajets


In [ ]:
df_journeys_raw.filter(
    pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=14),
).group_by(["driver_identity_key", pl.col("start_datetime").dt.truncate("1w")]).agg(
    pl.len().alias("num_journeys"),
    pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
    .n_unique()
    .alias("num_trips"),
).group_by(["start_datetime"]).agg(
    pl.col("num_journeys").mean().alias("Nombre moyen de journeys par semaine"),
    pl.col("num_trips").mean().alias("Nombre moyen de trips par semaine"),
).select(
    pl.col("Nombre moyen de journeys par semaine").mean(),
    pl.col("Nombre moyen de trips par semaine").mean(),
)

In [ ]:
df_journeys.filter(
    pl.col("first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
    pl.col("start_datetime") <= pl.col("first_trip_datetime") + pl.duration(days=30),
).group_by(["driver_identity_key"]).agg(
    pl.len().alias("num_journeys"),
    pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
    .n_unique()
    .alias("num_trips"),
).select(
    pl.col("num_journeys").mean().alias("Nombre moyen de journeys sur 30 jours"),
    pl.col("num_trips").mean().alias("Nombre moyen de trips sur 30 jours"),
)

In [ ]:
def create_trip_distribution_chart(
    df: pl.DataFrame,
    entity_type: str,  
    labels_map: dict,
    start_date: datetime =  datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    breaks: range = range(1, 31, 2),
    period_days: int = 30,
) -> go.Figure:
    """
    Crée un histogramme de distribution des trajets par entité.
    
    Args:
        df: DataFrame Polars
        entity_type: "driver" ou "passenger"
        start_date: Date de début du filtre
        labels_map: Dictionnaire de mapping des labels
        breaks: Range pour les bins de l'histogramme
        period_days: Nombre de jours de la période d'analyse
    
    Returns:
        Figure plotly
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    identity_col = config["identity_col"]
    first_trip_col = config["first_trip_col"]
    entity_label = config["label_plural"]
    
    data_agg = (
        df.filter(
            pl.col(first_trip_col) >= start_date,
            pl.col(first_trip_col) <= datetime.now(ZoneInfo("GMT")) - timedelta(days=period_days),
            pl.col("start_datetime") <= pl.col(first_trip_col) + pl.duration(days=period_days),
        )
        .group_by([identity_col])
        .agg(
            pl.len().alias("num_journeys"),
            pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
            .n_unique()
            .alias("num_trips"),
        )
        .with_columns(
            pl.col("num_trips").cut(breaks=breaks, left_closed=True, include_breaks=True)
        )
        .group_by(["num_trips"])
        .agg(pl.col(identity_col).n_unique().alias(f"num_{entity_type}s"))
        .with_columns(
            pl.col("num_trips").struct.unnest(),
            (100 * pl.col(f"num_{entity_type}s") / pl.col(f"num_{entity_type}s").sum())
            .round(2)
            .alias(f"share_{entity_type}s"),
        )
        .sort("breakpoint")
    )
    
    fig = px.bar(
        data_agg,
        x=data_agg["category"],
        y=data_agg[f"share_{entity_type}s"],
        labels={**labels_map},
        text=f"share_{entity_type}s",
        text_auto=".1f",
        template="simple_white",
        title=f"Distribution du nombre de {entity_label} en fonction du nombre de trajets effectués <br><sub>Période de {period_days} jours</sub>",
    )
    
    return fig

In [ ]:


fig_drivers_by_trip_numbers_hist = create_trip_distribution_chart(
    df=df_journeys_raw,
    entity_type="driver",
    start_date=datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    labels_map=labels_map,
)

fig_drivers_by_trip_numbers_hist.update_yaxes(showgrid=True)
fig_drivers_by_trip_numbers_hist.show()

fig_drivers_by_trip_numbers_hist.write_html(
    OUTPUT_PATH / "fig_histo_trajets_conducteurs.html"
)
fig_drivers_by_trip_numbers_hist.write_image(
    OUTPUT_PATH / "fig_histo_trajets_conducteurs.svg", width=1280, height=720
)

In [ ]:
fig_drivers_by_trip_numbers_hist = create_trip_distribution_chart(
    df=df_journeys_raw,
    entity_type="driver",
    start_date=datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    labels_map=labels_map,
    breaks= [1, 61, 1000],
)

fig_drivers_by_trip_numbers_hist.update_yaxes(showgrid=True)
fig_drivers_by_trip_numbers_hist.show()

fig_drivers_by_trip_numbers_hist.write_html(
    OUTPUT_PATH / "fig_histo_trajets_conducteurs.html"
)
fig_drivers_by_trip_numbers_hist.write_image(
    OUTPUT_PATH / "fig_histo_trajets_conducteurs.svg", width=1280, height=720
)

## Rétention


In [ ]:
def create_retention_analysis(
    df: pl.DataFrame,
    entity_type: str,
    start_date: datetime = DEFAULT_START_DATE,
    num_weeks: int = 12,
    group_by_col: str | None = None,
) -> pl.DataFrame:
    """
    Analyse la rétention des entités semaine par semaine avec groupement optionnel.
    
    Args:
        df: DataFrame Polars
        entity_type: "driver" ou "passenger"
        start_date: Date de début du filtre
        num_weeks: Nombre de semaines à analyser après la première course
        group_by_col: Colonne optionnelle pour grouper (ex: "is_near_station_fmt")
    
    Returns:
        DataFrame avec week_number, share_{entity_type}s, et group_by_col si fourni
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    identity_col = config["identity_col"]
    first_trip_col = config["first_trip_col"]
    
    initial_group_cols = [identity_col]
    if group_by_col:
        initial_group_cols.append(group_by_col)
    
    partition_cols = [identity_col]
    if group_by_col:
        partition_cols.append(group_by_col)
    
    final_group_cols = ["week_number"]
    if group_by_col:
        final_group_cols.append(group_by_col)
    
    df_retention = (
        df.filter(
            pl.col(first_trip_col) >= start_date,
            pl.col(first_trip_col) <= datetime.now(ZoneInfo("GMT")) - timedelta(weeks=num_weeks),
        )
        .group_by(initial_group_cols)
        .agg(
            pl.col("start_datetime").min(),
            pl.datetime_range(
                pl.col("start_datetime").min().dt.truncate("1w"),
                pl.col("start_datetime").min().dt.truncate("1w") + pl.duration(weeks=num_weeks),
                "1w",
            ).alias("week"),
        )
        .explode("week")
        .join(
            df.filter(
                pl.col("start_datetime") >= start_date,
            ),
            left_on=[identity_col, "week"] + ([group_by_col] if group_by_col else []),
            right_on=[
                identity_col,
                pl.col("start_datetime").dt.truncate("1w"),
            ] + ([group_by_col] if group_by_col else []),
            how="left",
        )
        .group_by([pl.col(identity_col), "week"] + ([group_by_col] if group_by_col else []))
        .agg(
            (pl.col("_id").count() > 0).alias("has_traveled"),
        )
        .with_columns(
            pl.col("week")
            .rank()
            .over(partition_by=partition_cols, order_by="week")
            .alias("week_number")
        )
        .group_by(final_group_cols)
        .agg(
            (100 * pl.col("has_traveled").sum() / pl.col("has_traveled").count()).alias(
                f"{entity_type}s_share"
            )
        )
        .sort(["week_number"] + ([group_by_col] if group_by_col else []))
    )
    
    return df_retention


def plot_retention_analysis(
    df_retention: pl.DataFrame,
    entity_type: str,
    labels_map: dict = None,
    num_weeks: int = 12,
    group_by_col: str | None = None,
    color_discrete_map: dict | None = None,
) -> go.Figure:
    """
    Crée un graphique de rétention à partir du DataFrame d'analyse avec groupement optionnel.
    
    Args:
        df_retention: DataFrame résultat de create_retention_analysis
        entity_type: "driver" ou "passenger"
        labels_map: Dictionnaire de mapping des labels
        num_weeks: Nombre de semaines analysées
        group_by_col: Colonne optionnelle pour grouper (ex: "is_near_station_fmt")
        color_discrete_map: Mapping des couleurs pour le groupement
    
    Returns:
        Figure plotly
    """
    if entity_type not in ENTITY_CONFIGS:
        raise ValueError(f"entity_type doit être dans {list(ENTITY_CONFIGS.keys())}")
    
    config = ENTITY_CONFIGS[entity_type]
    entity_label = config["label_plural"]
    
    if labels_map is None:
        labels_map = {}
    
    chart_labels = {
        **labels_map,
        "week_number": "Numéro de semaine",
        f"{entity_type}s_share": f"% de {entity_label} actifs"
    }
    
    fig_kwargs = {
        "data_frame": df_retention,
        "x": "week_number",
        "y": f"{entity_type}s_share",
        "labels": chart_labels,
        "markers": True,
        "template": "simple_white",
        "title": f"Rétention des {entity_label} sur {num_weeks} semaines <br><sub>% de {entity_label} ayant effectué au moins un trajet par semaine</sub>",
    }
    
    if group_by_col:
        fig_kwargs["color"] = group_by_col
        if color_discrete_map:
            fig_kwargs["color_discrete_map"] = color_discrete_map
    
    fig = px.line(**fig_kwargs)
    
    fig.add_hline(y=100, line_dash="dash", line_color="gray", opacity=0.5)
    
    return fig

In [ ]:
NUM_WEEKS = 3 * 4
df_acquisition_by_driver_type = create_retention_analysis(
    df=df_journeys,
    entity_type="driver",
    start_date=datetime(2025, 2, 1, tzinfo=ZoneInfo("GMT")),
    num_weeks=NUM_WEEKS
)
df_acquisition_by_driver_type

In [ ]:
fig_churn_by_campaign_type = plot_retention_analysis(
    df_retention=df_acquisition_by_driver_type,
    entity_type="driver",
    labels_map=labels_map,
    num_weeks=NUM_WEEKS
)

fig_churn_by_campaign_type.update_yaxes(showgrid=True)
fig_churn_by_campaign_type.show()

fig_churn_by_campaign_type.write_html(OUTPUT_PATH / "fig_attrition_conducteur.html")
fig_churn_by_campaign_type.write_image(
    OUTPUT_PATH / "fig_attrition_conducteur.svg", width=1280, height=720
)

# Passagers


In [ ]:
print(
    f"Nombre de passagers uniques : {
        df_journeys_raw.select(pl.col('passenger_identity_key').n_unique()).item()
    }"
)

## Acquisition

In [ ]:
fig_new_passenger_count_by_week = create_acquisition_chart(
    df=df_journeys_raw,
    entity_type="passenger",
    start_date=datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    labels_map=labels_map
)

fig_new_passenger_count_by_week.update_yaxes(showgrid=True)

fig_new_passenger_count_by_week.show()

fig_new_passenger_count_by_week.write_html(
    OUTPUT_PATH / "fig_new_passenger_count_by_week.html"
)
fig_new_passenger_count_by_week.write_image(
    OUTPUT_PATH / "fig_new_passenger_count_by_week.svg", width=1280, height=720
)

## Nombre de trajet

In [ ]:


df_journeys_raw.filter(
    pl.col("passenger_first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("passenger_first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=14),
).group_by(["passenger_identity_key", pl.col("start_datetime").dt.truncate("1w")]).agg(
    pl.len().alias("num_journeys"),
    pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
    .n_unique()
    .alias("num_trips"),
).group_by(["start_datetime"]).agg(
    pl.col("num_journeys").mean().alias("Nombre moyen de journeys par semaine"),
    pl.col("num_trips").mean().alias("Nombre moyen de trips par semaine"),
).select(
    pl.col("Nombre moyen de journeys par semaine").mean(),
    pl.col("Nombre moyen de trips par semaine").mean(),
)



In [ ]:


df_journeys.filter(
    pl.col("passenger_first_trip_datetime") >= datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    pl.col("passenger_first_trip_datetime") <= datetime.now(ZoneInfo("GMT")) - timedelta(days=30),
    pl.col("start_datetime") <= pl.col("first_trip_datetime") + pl.duration(days=30),
).group_by(["passenger_identity_key"]).agg(
    pl.len().alias("num_journeys"),
    pl.concat_str(pl.col("operator_id"), pl.lit("-"), pl.col("operator_trip_id"))
    .n_unique()
    .alias("num_trips"),
).select(
    pl.col("num_journeys").mean().alias("Nombre moyen de journeys sur 30 jours"),
    pl.col("num_trips").mean().alias("Nombre moyen de trips sur 30 jours"),
)



In [ ]:

fig_passengers_by_trip_numbers_hist = create_trip_distribution_chart(
    df=df_journeys_raw,
    entity_type="passenger",
    start_date=datetime(2024, 9, 1, tzinfo=ZoneInfo("GMT")),
    labels_map=labels_map
)

fig_passengers_by_trip_numbers_hist.update_yaxes(showgrid=True)
fig_passengers_by_trip_numbers_hist.show()

fig_passengers_by_trip_numbers_hist.write_html(
    OUTPUT_PATH / "fig_histo_trajets_passager.html"
)
fig_passengers_by_trip_numbers_hist.write_image(
    OUTPUT_PATH / "fig_histo_trajets_passagers.svg", width=1280, height=720
)


## Attrition

In [ ]:
NUM_WEEKS = 3 * 4
df_acquisition_passenger = create_retention_analysis(
    df=df_journeys,
    entity_type="passenger",
    start_date=datetime(2025, 2, 1, tzinfo=ZoneInfo("GMT")),
    num_weeks=NUM_WEEKS
)
df_acquisition_passenger


In [ ]:
fig_churn_passenger = plot_retention_analysis(
    df_retention=df_acquisition_passenger,
    entity_type="passenger",
    labels_map=labels_map,
    num_weeks=NUM_WEEKS
)

fig_churn_passenger.update_yaxes(showgrid=True)
fig_churn_passenger.show()

fig_churn_passenger.write_html(OUTPUT_PATH / "fig_attrition_passager.html")
fig_churn_passenger.write_image(
    OUTPUT_PATH / "fig_attrition_passager.svg", width=1280, height=720
)


## Pass Navigo


In [ ]:
df_passengers_navigo = df_journeys_raw.group_by("passenger_identity_key").agg(
    pl.col("passenger_travelpass_name").max(),
    pl.col("passenger_travelpass_user_id").max(),
)
df_passengers_navigo

In [ ]:
f"Nombre de passagers avec un pass navigo : {
    df_passengers_navigo.select(
        pl.col('passenger_travelpass_user_id').is_not_null().sum()
    ).item()
}"

**NOTE** : J'ai identifié que il n'y a pas de trajets avec le pass navigo renseigné depuis septembre 2020 si on exclut 9 trajets en 2025


# Carto

In [ ]:
def get_all_trips_with_time_slot(df_journeys_raw: pl.DataFrame):
    return (
        df_journeys_raw
        .with_columns([
            pl.col("start_datetime").dt.hour().alias("hour"),
            pl.when(pl.col("start_datetime").dt.hour().is_between(6, 9))
            .then(pl.lit("Matin (6h-9h)"))
            .when(pl.col("start_datetime").dt.hour().is_between(9, 12))
            .then(pl.lit("Matinée (9h-12h)"))
            .when(pl.col("start_datetime").dt.hour().is_between(12, 16))
            .then(pl.lit("Midi (12h-14h)"))
            .when(pl.col("start_datetime").dt.hour().is_between(16, 19))
            .then(pl.lit("Après-midi (14h-16h)"))
            .when(pl.col("start_datetime").dt.hour().is_between(18, 21))
            .then(pl.lit("Soir (16h-19h)"))
            .otherwise(pl.lit("Nuit (19h-6h)"))
            .alias("time_slot")
        ])
    )
def pl_add_h3cells_wgrpby(df_journeys_raw: pl.DataFrame, end_or_start: str, group_by_cols:list[str]):
    group_by_cols = group_by_cols + ["h3_cell"]
    df_h3 = (
    df_journeys_raw.with_columns(
        plh3.latlng_to_cell(
            lat=f"{end_or_start}_latitude", lng=f"{end_or_start}_longitude", resolution=8
        ).alias("h3_cell"),
    ).group_by(group_by_cols
            ).agg(pl.col("_id").n_unique().alias("num_journeys")
            )
)
    return df_h3

def df_add_departure_arrival_delta(df_start: pl.DataFrame, df_end: pl.DataFrame):
    joined_df = (
        df_start.join(
            df_end,
            on=["h3_cell", "time_slot"],
            how="full",
            validate="1:1",
        )
        .with_columns([
            # Unifier les colonnes après le full join
            pl.coalesce(["h3_cell", "h3_cell_right"]).alias("cell_joined"),
            pl.coalesce(["time_slot", "time_slot_right"]).alias("time_slot_unified"),
            # Remplir les NULL et caster en Int64 
            pl.col("num_journeys").fill_null(0).cast(pl.Int64).alias("num_starts"),
            pl.col("num_journeys_right").fill_null(0).cast(pl.Int64).alias("num_ends"),
        ])
        .with_columns([
            # Calculer le delta en pourcentage
            pl.when((pl.col("num_starts") + pl.col("num_ends")) == 0)
            .then(pl.lit(0.0))
            .otherwise(
                (pl.col("num_starts") - pl.col("num_ends")).cast(pl.Float64) * 100.0
                / (pl.col("num_starts") + pl.col("num_ends"))
            ).alias("delta"),
            # Créer la géométrie
            plh3.cell_to_boundary(pl.col("cell_joined")).alias("cell_geom"),
        ])
        .with_columns([
            # Convertir en Polygon Shapely
            pl.col("cell_geom").map_elements(
                lambda x: shapely.Polygon([[e[1], e[0]] for e in x]),
                return_dtype=pl.Object
            ).alias("cell_geom"),
            pl.col("cell_joined").cast(pl.String),
        ])
        .drop(["h3_cell", "h3_cell_right", "time_slot", "time_slot_right", "num_journeys", "num_journeys_right"])
        .rename({"time_slot_unified": "time_slot"})
    )
    return joined_df


In [ ]:

def create_differential_density_maps(df_journeys_raw: pl.DataFrame):
    """Crée une carte différentielle par tranche horaire avec H3"""
    data = get_all_trips_with_time_slot(df_journeys_raw)
    df_h3_start = pl_add_h3cells_wgrpby(data, "start", ["time_slot"])
    df_h3_end = pl_add_h3cells_wgrpby(data, "end", ["time_slot"])
    df_h3 = df_add_departure_arrival_delta(df_h3_start, df_h3_end)
    print(df_h3.head())
    gdf = gpd.GeoDataFrame(
        df_h3.to_pandas()
    ).set_geometry("cell_geom", crs=4326) 
    
    # Créer une carte par tranche horaire
    time_slots = sorted(df_h3["time_slot"].unique().to_list())
    figs = {}
    
    for slot in time_slots:
        gdf_slot = gdf[gdf["time_slot"] == slot]
        
        if len(gdf_slot) == 0:
            continue
        
        center = gdf_slot.geometry.unary_union.centroid.coords[0][::-1]  # (lat, lon)
        
        cmap = cm.get_cmap("PiYG")
        color_scale = bcm.LinearColormap(
            colors=[mcolors.to_hex(cmap(i)) for i in [0.0, 0.25, 0.5, 0.75, 1.0]],
            vmin=-100,
            vmax=100,
        )
        color_scale.caption = "Delta départ/arrivée (%)"
        
        m = folium.Map(location=center, zoom_start=9, tiles="openstreetmap")
        
        folium.GeoJson(
            gdf_slot.to_json(), 
            style_function=lambda feature: {
                "fillColor": color_scale(feature["properties"]["delta"]),  
                "color": "black",
                "weight": 1,
                "fillOpacity": 0.5,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=["cell_joined", "num_starts", "num_ends", "delta"],
                aliases=["Zone H3", "Départs", "Arrivées", "Delta (%)"],
                localize=True,
            ),
        ).add_to(m)
        
        # Ajout de la colorbar à la carte
        color_scale.add_to(m)
        
        figs[slot] = m
    
    return figs

# Exécution
diff_maps = create_differential_density_maps(df_journeys_raw)
for slot, fig in diff_maps.items():
    filename = slot.replace(" ", "_").replace("(", "").replace(")", "").replace("️", "")
    fig.save(OUTPUT_PATH / f"diff_density_h3_{filename}.html")
